# Ensemble Learning Tutorial

**Comparing Multiple Machine Learning Models on Classification and Regression Tasks**

This notebook is based on the `ensemble.py` script and demonstrates how to compare several models:

- Neural Network (MLP)
- Decision Tree
- Random Forest
- AdaBoost
- Gradient Boosting
- XGBoost

We will run experiments on two problems:
1. **Classification** — Pima Indians Diabetes dataset
2. **Regression** — Energy Efficiency dataset (Heating Load)

---

## 1. Import Required Libraries

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from numpy import genfromtxt
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, accuracy_score, confusion_matrix
from sklearn.preprocessing import Normalizer

# Scikit-learn models
from sklearn.neural_network import MLPClassifier, MLPRegressor
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor, export_text
from sklearn.ensemble import (
    RandomForestClassifier, RandomForestRegressor,
    AdaBoostClassifier, AdaBoostRegressor,
    GradientBoostingClassifier, GradientBoostingRegressor
)

import xgboost as xgb

import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8')
%matplotlib inline

ModuleNotFoundError: No module named 'xgboost'

## 2. Data Loading Function

In [2]:
def read_data(run_num=42, prob='classification'):
    """
    Load either classification or regression dataset
    """
    if prob == 'classification':
        # Pima Indians Diabetes Dataset
        data_in = genfromtxt("data/pima.csv", delimiter=",")
        X = data_in[:, 0:8]
        y = data_in[:, -1]
        print("Loaded Pima Indians Diabetes Dataset (Classification)")
        
    elif prob == 'regression':
        # Energy Efficiency Dataset
        data_in = genfromtxt('data/ENB2012_data.csv', delimiter=",")
        X = data_in[:, 0:8]
        y = data_in[:, 8]  # Heating Load
        print("Loaded Energy Efficiency Dataset (Regression - Heating Load)")
    
    # Optional normalization
    # transformer = Normalizer().fit(X)
    # X = transformer.transform(X)
    
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.40, random_state=run_num
    )
    
    return X_train, X_test, y_train, y_test

## 3. Model Training & Evaluation Function

In [3]:
def train_and_evaluate(X_train, X_test, y_train, y_test, 
                      model_type, hidden=8, learn_rate=0.01, run_num=0, problem='classification'):
    """
    Train a single model and return performance
    """
    tree_depth = 2
    
    if problem == 'classification':
        if model_type == 'MLP':
            model = MLPClassifier(hidden_layer_sizes=(hidden,), 
                                random_state=run_num, max_iter=100, 
                                solver='sgd', learning_rate_init=learn_rate)
            
        elif model_type == 'DecisionTree':
            model = DecisionTreeClassifier(random_state=0, max_depth=tree_depth)
            
        elif model_type == 'RandomForest':
            model = RandomForestClassifier(n_estimators=100, max_depth=tree_depth, random_state=run_num)
            
        elif model_type == 'AdaBoost':
            model = AdaBoostClassifier(n_estimators=100, random_state=run_num)
            
        elif model_type == 'GradientBoosting':
            model = GradientBoostingClassifier(n_estimators=10, random_state=run_num)
            
        elif model_type == 'XGBoost':
            model = xgb.XGBClassifier(colsample_bytree=0.3, learning_rate=0.1,
                                    max_depth=5, alpha=5, n_estimators=100, random_state=run_num)
    
    else:  # regression
        if model_type == 'MLP':
            model = MLPRegressor(hidden_layer_sizes=(hidden*3,), 
                               random_state=run_num, max_iter=500, 
                               solver='adam', learning_rate_init=learn_rate)
            
        elif model_type == 'DecisionTree':
            model = DecisionTreeRegressor(random_state=0, max_depth=tree_depth)
            
        elif model_type == 'RandomForest':
            model = RandomForestRegressor(n_estimators=100, max_depth=tree_depth, random_state=run_num)
            
        elif model_type == 'AdaBoost':
            model = AdaBoostRegressor(n_estimators=100, random_state=run_num)
            
        elif model_type == 'GradientBoosting':
            model = GradientBoostingRegressor(n_estimators=10, random_state=run_num)
            
        elif model_type == 'XGBoost':
            model = xgb.XGBRegressor(objective='reg:squarederror', colsample_bytree=0.3, 
                                   learning_rate=0.1, max_depth=5, alpha=5, n_estimators=100, 
                                   random_state=run_num)
    
    # Train the model
    model.fit(X_train, y_train)
    
    # Predictions
    y_pred_test = model.predict(X_test)
    y_pred_train = model.predict(X_train)
    
    # Performance metrics
    if problem == 'regression':
        perf_test = np.sqrt(mean_squared_error(y_test, y_pred_test))
        perf_train = np.sqrt(mean_squared_error(y_train, y_pred_train))
        metric_name = 'RMSE'
    else:
        perf_test = accuracy_score(y_test, y_pred_test)
        perf_train = accuracy_score(y_train, y_pred_train)
        metric_name = 'Accuracy'
    
    return perf_test, perf_train, metric_name

## 4. Run Multiple Experiments

In [4]:
def run_experiments(problem='classification', max_runs=10):
    """
    Run multiple experiments with different random seeds
    """
    print(f"Running {max_runs} experiments for {problem.upper()} problem...\n")
    
    models = ['MLP', 'DecisionTree', 'RandomForest', 'AdaBoost', 'GradientBoosting', 'XGBoost']
    
    results = {model: [] for model in models}
    
    for run in range(max_runs):
        print(f"Run {run+1}/{max_runs} (seed={run})")
        X_train, X_test, y_train, y_test = read_data(run_num=run, prob=problem)
        
        for model_name in models:
            test_score, _, _ = train_and_evaluate(
                X_train, X_test, y_train, y_test,
                model_type=model_name,
                hidden=8,
                learn_rate=0.01,
                run_num=run,
                problem=problem
            )
            results[model_name].append(test_score)
    
    # Convert to DataFrame for easy analysis
    df_results = pd.DataFrame(results)
    
    print("\n" + "="*60)
    print(f"FINAL RESULTS - {problem.upper()}")
    print("="*60)
    
    summary = pd.DataFrame({
        'Mean': df_results.mean(),
        'Std': df_results.std(),
        'Min': df_results.min(),
        'Max': df_results.max()
    })
    
    print(summary.round(4))
    
    return df_results, summary

## 5. Run Classification Experiment

In [5]:
# Classification Experiment
df_classif, summary_classif = run_experiments(problem='classification', max_runs=5)

Running 5 experiments for CLASSIFICATION problem...

Run 1/5 (seed=0)
Loaded Pima Indians Diabetes Dataset (Classification)


NameError: name 'xgb' is not defined

## 6. Run Regression Experiment

In [ ]:
# Regression Experiment
df_reg, summary_reg = run_experiments(problem='regression', max_runs=5)

## 7. Visualization of Results

In [ ]:
def plot_results(df, title, metric):
    plt.figure(figsize=(12, 6))
    
    # Boxplot
    sns.boxplot(data=df, palette="Set3")
    plt.title(f'{title} - {metric} Comparison', fontsize=16)
    plt.ylabel(metric)
    plt.xticks(rotation=45)
    plt.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.show()
    
    # Bar plot of means
    plt.figure(figsize=(10, 5))
    means = df.mean().sort_values(ascending=False)
    sns.barplot(x=means.index, y=means.values, palette="viridis")
    plt.title(f'{title} - Mean {metric} (Higher is better for Accuracy, Lower for RMSE)', fontsize=14)
    plt.ylabel(metric)
    plt.xticks(rotation=45)
    plt.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.show()

In [ ]:
# Plot Classification Results
plot_results(df_classif, 'Classification (Pima Diabetes)', 'Accuracy')

In [ ]:
# Plot Regression Results
plot_results(df_reg, 'Regression (Energy Efficiency)', 'RMSE')

## 8. Summary & Observations

**Key Takeaways:**

- **Ensemble methods** (Random Forest, Gradient Boosting, XGBoost, AdaBoost) usually outperform single models like Decision Tree and basic Neural Networks.
- **XGBoost** often achieves excellent performance with minimal tuning.
- **Random Forest** is very robust and less sensitive to hyperparameters.
- **Neural Networks** (MLP) can be competitive but often require more careful tuning of learning rate, hidden layers, and number of epochs.
- Performance varies significantly across different random seeds → always run multiple experiments!

---

**Next Steps to Try:**
1. Increase `max_runs` to 20 or 30 for more stable statistics
2. Try hyperparameter tuning (GridSearchCV or Optuna)
3. Add feature importance analysis for tree-based models
4. Experiment with stacking ensembles
5. Try different datasets

Would you like me to add sections on **feature importance**, **confusion matrix visualization**, or **hyperparameter tuning**?